# WanGP Kaggle Testing Notebook

Run these cells from top to bottom in a Kaggle notebook with GPU enabled. The repo, model checkpoints, Hugging Face cache, Torch cache, pip cache, and temp files are placed under `/kaggle/temp` so `/kaggle/working` is only used for final outputs.

## Cell 1 - Check GPU And Disk

In [ ]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null || df -h

## Cell 2 - Configure Paths

Change `REPO_URL` if you want to test a different fork or branch. Keep the large paths in temp unless you intentionally want Kaggle to persist them.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/SohailAwg/wan2gp_studio.git"
BRANCH = "main"

TEMP_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")
ROOT = TEMP_ROOT / "Wan2GP-main"
MODEL_DIR = TEMP_ROOT / "wan2gp_models"
CACHE_DIR = TEMP_ROOT / "wan2gp_cache"
OUTPUT_DIR = Path("/kaggle/working/wan2gp_outputs")

# Set False after the first successful install in the same Kaggle session.
INSTALL_DEPS = True

# Safe first-run settings for Kaggle. If Sage/Flash attention is installed later, you can try changing ATTENTION.
ATTENTION = "sdpa"
PROFILE = "4"
PORT = 7860
USE_GRADIO_SHARE = False
DISABLE_BUNDLED_PLUGINS = False
WAN2GP_WAIT_SECONDS = 1800
STOP_SERVER_AT_END = False

for path in [TEMP_ROOT, MODEL_DIR, CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo:", ROOT)
print("Models/checkpoints:", MODEL_DIR)
print("Caches:", CACHE_DIR)
print("Persisted outputs:", OUTPUT_DIR)

## Cell 3 - Force Large Caches Into Temp

Run this before installing packages or importing WanGP.

In [ ]:
import os

cache_map = {
    "HF_HOME": CACHE_DIR / "huggingface",
    "HUGGINGFACE_HUB_CACHE": CACHE_DIR / "huggingface" / "hub",
    "TRANSFORMERS_CACHE": CACHE_DIR / "huggingface" / "transformers",
    "DIFFUSERS_CACHE": CACHE_DIR / "huggingface" / "diffusers",
    "TORCH_HOME": CACHE_DIR / "torch",
    "XDG_CACHE_HOME": CACHE_DIR / "xdg",
    "PIP_CACHE_DIR": CACHE_DIR / "pip",
    "GRADIO_TEMP_DIR": TEMP_ROOT / "gradio_tmp",
    "MPLBACKEND": "Agg",
    "TMPDIR": TEMP_ROOT / "tmp",
    "TEMP": TEMP_ROOT / "tmp",
    "TMP": TEMP_ROOT / "tmp",
}

for key, value in cache_map.items():
    if isinstance(value, Path):
        value.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(value)

# Avoid writing tokenizers parallelism warnings into long logs.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["WAN2GP_KAGGLE_MODE"] = "1"
if globals().get("DISABLE_BUNDLED_PLUGINS", True):
    os.environ["WAN2GP_DISABLE_BUNDLED_PLUGINS"] = "1"
else:
    os.environ.pop("WAN2GP_DISABLE_BUNDLED_PLUGINS", None)

for key in cache_map:
    print(f"{key}={os.environ[key]}")

## Cell 4 - Clone Or Update WanGP In Temp

In [ ]:
import os
import subprocess
import shutil

if (ROOT / ".git").exists():
    os.chdir(ROOT)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "clean", "-fd"], check=True)
else:
    if ROOT.exists():
        print(f"Removing incomplete non-git folder: {ROOT}")
        shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)

print("Current repo:", ROOT)
!git -C "$ROOT" rev-parse --short HEAD

## Cell 5 - Link Checkpoints To Temp And Outputs To Working

WanGP normally uses `ckpts` for downloaded model files. This cell makes that folder point at `/kaggle/temp/wan2gp_models`.

In [ ]:
import os
import shutil
from pathlib import Path

def replace_with_symlink(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        if link_path.resolve() == target_path.resolve():
            return
        link_path.unlink()
    elif link_path.exists():
        backup = link_path.with_name(link_path.name + "_backup_in_temp")
        if backup.exists():
            shutil.rmtree(backup) if backup.is_dir() else backup.unlink()
        shutil.move(str(link_path), str(backup))
        print(f"Moved existing {link_path} to {backup}")
    os.symlink(target_path, link_path, target_is_directory=True)

replace_with_symlink(ROOT / "ckpts", MODEL_DIR)
replace_with_symlink(ROOT / "outputs", OUTPUT_DIR)

os.chdir(ROOT)
print("Working directory:", Path.cwd())
print("ckpts ->", (ROOT / "ckpts").resolve())
print("outputs ->", (ROOT / "outputs").resolve())

# Kaggle is headless. This bundled optional mask tool forces TkAgg unless patched.
matanyone_file = ROOT / "preprocessing" / "matanyone" / "tools" / "interact_tools.py"
if matanyone_file.exists():
    text = matanyone_file.read_text(encoding="utf-8")
    patched = text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
    if patched != text:
        matanyone_file.write_text(patched, encoding="utf-8")
        print("Patched MatAnyone matplotlib backend for Kaggle headless mode.")

required_assets = [
    ROOT / "plugins" / "wan2gp-motion-designer" / "assets" / "motion_designer_iframe_template.html",
    ROOT / "plugins" / "wan2gp-motion-designer" / "assets" / "app.js",
    ROOT / "plugins" / "wan2gp-motion-designer" / "assets" / "style.css",
]
missing_assets = [str(path) for path in required_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError("Missing required WanGP UI asset(s). Pull the latest repo before continuing: " + ", ".join(missing_assets))
print("Required WanGP UI assets found.")

## Cell 6 - Install Dependencies

This can take a while. It uses `--no-cache-dir` and the temp pip cache so package downloads do not consume `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys

os.chdir(ROOT)

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "--no-cache-dir"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-r", "requirements.txt"], check=True)
else:
    print("Skipping dependency install because INSTALL_DEPS=False")

## Cell 7 - Optional Hugging Face Login

If you need gated models, add a Kaggle secret named `HF_TOKEN` first. Public models can download without this.

In [ ]:
import os

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for name in ["HF_TOKEN", "HUGGINGFACE_TOKEN"]:
            try:
                token = secrets.get_secret(name)
                if token:
                    os.environ["HF_TOKEN"] = token
                    break
            except Exception:
                pass
    except Exception as exc:
        print("Kaggle secrets are not available in this environment:", exc)

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF_TOKEN loaded and Hugging Face login completed.")
else:
    print("No HF_TOKEN found. Public models may still work; gated models will fail until you add the secret.")

## Cell 8 - Launch WanGP UI

This starts WanGP under a small restart supervisor and prints the Gradio public URL when it appears. If the UI requests a restart, the supervisor relaunches WanGP on the same port so the Cloudflare URL can recover. First model use may download many GB into `/kaggle/temp/wan2gp_models`.

In [ ]:
import os
import re
import socket
import subprocess
import sys
import threading
import time

os.chdir(ROOT)

try:
    if WAN2GP_PROC.poll() is None:
        WAN2GP_PROC.terminate()
        time.sleep(5)
except NameError:
    pass

cmd = [
    sys.executable,
    "wgp.py",
    "--listen",
    "--server-port", str(PORT),
    "--attention", ATTENTION,
    "--profile", PROFILE,
    "--perc-reserved-mem-max", "0.25",
]
if globals().get("USE_GRADIO_SHARE", False):
    cmd.insert(2, "--share")

supervisor_path = TEMP_ROOT / "wan2gp_kaggle_supervisor.py"
supervisor_path.write_text("""
import os
import signal
import subprocess
import sys
import time

cmd = sys.argv[1:]
child = None
stopping = False

def request_stop(signum, frame):
    global stopping
    stopping = True
    if child is not None and child.poll() is None:
        child.terminate()

signal.signal(signal.SIGTERM, request_stop)
signal.signal(signal.SIGINT, request_stop)

while True:
    print('[Kaggle supervisor] Starting WanGP:', ' '.join(cmd), flush=True)
    child = subprocess.Popen(cmd, cwd=os.getcwd(), env=os.environ.copy())
    code = child.wait()
    print(f'[Kaggle supervisor] WanGP exited with code {code}', flush=True)
    if stopping:
        sys.exit(0)
    if code != 42:
        sys.exit(code if code is not None else 1)
    print('[Kaggle supervisor] Restart requested; relaunching in 5 seconds on the same port.', flush=True)
    time.sleep(5)
""", encoding="utf-8")

launch_cmd = [sys.executable, str(supervisor_path), *cmd]
print("Launching under Kaggle restart supervisor:", " ".join(launch_cmd))
print("WanGP command:", " ".join(cmd))
WAN2GP_LOG_LINES = []
WAN2GP_PROC = subprocess.Popen(
    launch_cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

def pump_logs():
    for line in WAN2GP_PROC.stdout:
        WAN2GP_LOG_LINES.append(line.rstrip())
        print(line, end="")

threading.Thread(target=pump_logs, daemon=True).start()

public_url = None
local_url = None
wait_seconds = int(globals().get("WAN2GP_WAIT_SECONDS", 1800))
started_at = time.time()
last_heartbeat = 0
local_ready_seen_at = None

for _ in range(wait_seconds):
    text = "\n".join(WAN2GP_LOG_LINES[-80:])
    try:
        with socket.create_connection(("127.0.0.1", int(PORT)), timeout=1):
            local_url = f"http://127.0.0.1:{PORT}"
            if local_ready_seen_at is None:
                local_ready_seen_at = time.time()
    except OSError:
        pass
    match = re.search(r"https://[a-zA-Z0-9.-]+\.gradio\.live", text)
    if match:
        public_url = match.group(0)
        break
    local_match = re.search(r"http://(?:127\.0\.0\.1|0\.0\.0\.0|localhost):\d+", text)
    if local_match:
        local_url = local_match.group(0)
        if local_ready_seen_at is None:
            local_ready_seen_at = time.time()
    if WAN2GP_PROC.poll() is not None:
        break
    elapsed = int(time.time() - started_at)
    if local_url and local_ready_seen_at and time.time() - local_ready_seen_at >= 20:
        print("\nLocal Gradio server is ready; moving on to fallback tunnel if no gradio.live URL appeared.")
        break
    if elapsed - last_heartbeat >= 30:
        last_heartbeat = elapsed
        print(f"\nStill waiting for Gradio public URL... {elapsed}s elapsed, process is alive.")
    time.sleep(1)

if public_url:
    print("\nWanGP public URL:", public_url)
else:
    if local_url:
        print("\nNo gradio.live URL found yet; Cell 8B will create a Cloudflare public URL.")
    else:
        print("\nNo public URL found within", wait_seconds, "seconds.")
    print("Process return code:", WAN2GP_PROC.poll())
    if local_url:
        print("Local Gradio URL seen in logs:", local_url)
    print("Last startup log lines:")
    print("\n".join(WAN2GP_LOG_LINES[-40:]))

## Cell 8B - Fallback Public URL With Cloudflare

Run this if Cell 8 says the process is alive but no `gradio.live` URL appears. It creates a temporary `trycloudflare.com` URL for the local WanGP server. After clicking Save and Restart in WanGP, wait about 30 seconds and refresh the same public URL.

In [ ]:
import os
import re
import stat
import subprocess
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

if "WAN2GP_PROC" not in globals() or WAN2GP_PROC.poll() is not None:
    raise RuntimeError("WanGP is not running. Run Cell 8 first and wait until the process is alive.")

origin_url = f"http://127.0.0.1:{PORT}"
origin_ready = False
for _ in range(180):
    if WAN2GP_PROC.poll() is not None:
        raise RuntimeError(f"WanGP exited before the tunnel could start. Return code: {WAN2GP_PROC.poll()}")
    try:
        with urllib.request.urlopen(origin_url, timeout=3) as response:
            origin_ready = response.status < 500
    except urllib.error.HTTPError as exc:
        origin_ready = exc.code < 500
    except Exception:
        origin_ready = False
    if origin_ready:
        print("Local WanGP origin is responding:", origin_url)
        break
    time.sleep(1)

if not origin_ready:
    raise RuntimeError(f"WanGP is alive, but http://127.0.0.1:{PORT} did not respond. Do not use a public URL until the local origin is ready.")

cloudflared = TEMP_ROOT / "cloudflared"
if not cloudflared.exists():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    subprocess.run(["wget", "-q", "-O", str(cloudflared), url], check=True)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

try:
    if CLOUDFLARED_PROC.poll() is None:
        CLOUDFLARED_PROC.terminate()
        time.sleep(2)
except NameError:
    pass

CLOUDFLARED_LOG_LINES = []
CLOUDFLARED_PROC = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def pump_cloudflared_logs():
    for line in CLOUDFLARED_PROC.stdout:
        CLOUDFLARED_LOG_LINES.append(line.rstrip())
        print(line, end="")

threading.Thread(target=pump_cloudflared_logs, daemon=True).start()

cloudflare_url = None
for _ in range(120):
    text = "\n".join(CLOUDFLARED_LOG_LINES[-80:])
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if match:
        cloudflare_url = match.group(0)
        break
    if CLOUDFLARED_PROC.poll() is not None:
        break
    time.sleep(1)

if cloudflare_url:
    print("\nWanGP fallback public URL:", cloudflare_url)
    print("Keep this Kaggle session running. The URL stops working when WanGP or this notebook stops.")
else:
    print("\nNo Cloudflare URL found yet. Return code:", CLOUDFLARED_PROC.poll())
    print("\n".join(CLOUDFLARED_LOG_LINES[-40:]))

## Cell 9 - Check Space While Testing

In [ ]:
!du -sh "$ROOT" "$MODEL_DIR" "$CACHE_DIR" "$OUTPUT_DIR" 2>/dev/null || true
!df -h /kaggle/working /kaggle/temp /tmp 2>/dev/null || df -h

## Cell 10 - List Persisted Outputs

In [ ]:
from pathlib import Path

for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path, f"{path.stat().st_size / (1024 ** 2):.1f} MB")

## Cell 11 - Keep WanGP Running / Optional Stop

In [ ]:
if not globals().get("STOP_SERVER_AT_END", False):
    print("Leaving WanGP and the Cloudflare tunnel running.")
    print("Use the Cell 8B public URL while this Kaggle session stays active.")
    print("To stop intentionally, set STOP_SERVER_AT_END = True in Cell 2 and rerun this cell.")
else:
    for proc_name in ["CLOUDFLARED_PROC", "WAN2GP_PROC"]:
        proc = globals().get(proc_name)
        if proc is None:
            print(proc_name, "is not defined. Nothing to stop.")
            continue
        if proc.poll() is None:
            proc.terminate()
            print(proc_name, "stopped.")
        else:
            print(proc_name, "was already stopped with code", proc.poll())